# LC 76 — Minimum Window Substring
**Difficulty:** Hard &nbsp;|&nbsp; **Category:** String
&nbsp;|&nbsp; **Pattern:** Sliding Window — Two HashMap Counts

<div style="border-left: 4px solid purple; padding: 10px;
            background: #f9f0ff; margin: 10px 0;">
<strong>Core Insight:</strong> Expand the right pointer until
the window contains all required characters, then shrink
from the left to find the minimum valid window.
Track a <code>formed</code> counter so you know exactly
when the window becomes valid without re-scanning it.
</div>

## Official Problem Statement

Given two strings `s` and `t` of lengths `m` and `n`,
return the **minimum window substring** of `s` such that
every character in `t` (including duplicates) is included
in the window. If there is no such substring, return the
empty string `""`.

**Example 1:**
```
Input:  s = "ADOBECODEBANC", t = "ABC"
Output: "BANC"
```

**Example 2:**
```
Input:  s = "a", t = "a"
Output: "a"
```

**Example 3:**
```
Input:  s = "a", t = "aa"
Output: ""
```

**Constraints:**
- `m == s.length`
- `n == t.length`
- `1 <= m, n <= 10^5`
- `s` and `t` consist of uppercase and lowercase letters.
- The answer is **unique**.

## What This Is Actually Asking

You have a long string `s` and a short string `t`.
Find the shortest slice of `s` that contains every
letter from `t` — including any duplicates.
The letters in the slice do not need to be in order.
If no such slice exists, return an empty string.

## Walk Through an Example by Hand

Input: `s = "ADOBECODEBANC"`, `t = "ABC"`

Need: A=1, B=1, C=1 → required = 3 unique chars satisfied

Step 1 — expand right until window is valid:
  window = "ADOBEC" (indices 0-5)
  has A=1, B=1, C=1 → formed=3 ✓ — valid!

Step 2 — shrink left to minimize:
  remove A → window = "DOBEC" → formed=2 ✗ — invalid
  so best so far = "ADOBEC" (length 6)

Step 3 — expand right again:
  right moves to index 9 → window = "DOBECODEBA"
  has A=1, B=2, C=1 → formed=3 ✓ — valid!

Step 4 — shrink left:
  remove D  → "OBECODEBA"  — still valid
  remove O  → "BECODEBA"   — still valid
  remove B  → "ECODEBA"    — still valid (B still in window)
  remove E  → "CODEBA"     — still valid, length=6
  remove C  → "ODEBA"      → formed=2 ✗
  best so far = "CODEBA" (length 6)

Step 5 — expand right to index 12 (C):
  window = "ODEBANC" → formed=3 ✓
  shrink: remove O,D,E → "BANC" length=4 — new best!

Answer: "BANC"

## The Picture

```
s = A D O B E C O D E B A N C
    0 1 2 3 4 5 6 7 8 9 ...12

Sliding window squeezes from both ends:

Phase 1 — Expand R until window is valid:

  L                   R
  [ A  D  O  B  E  C ]
    has A, B, C → VALID

Phase 2 — Shrink L while window stays valid:

     L              R
     [ D  O  B  E  C ]
       missing A → INVALID — stop shrinking

Phase 3 — Expand R again to regain validity, repeat.

Final best window:

              L        R
              [ B A N C ]
                has A, B, C → VALID, length=4

Two HashMaps:
  need  = { A:1, B:1, C:1 }  ← what t requires
  window = { ... }            ← what current window has
  formed = count of chars where window[c] >= need[c]
```

## When To Use This Pattern

- When asked for the shortest/longest substring satisfying
  a character frequency condition.
- When the problem has an "expand until valid, shrink
  while valid" structure.
- When you need to track character counts across a moving
  range — two hashmaps fit naturally.
- When you see "window", "substring containing", or
  "minimum covering" in the problem.

## The Approach

Build a frequency map of what `t` needs.
Slide a window across `s`: expand the right edge one
character at a time, updating a window frequency map.
When every required character is satisfied (tracked by
a `formed` counter), try to shrink from the left
to find the smallest valid window.
Record the best (shortest) window seen at each valid state.

In [ ]:
from collections import Counter  # count char frequencies
from typing import Dict           # for type hints

In [ ]:
def test_harness(func):
    """Run all test cases against func and report results."""
    tests = [
        # (s, t, expected, label)
        (
            "ADOBECODEBANC", "ABC",
            "BANC",
            "LC example 1"
        ),
        (
            "a", "a",
            "a",
            "LC example 2 — single char match"
        ),
        (
            "a", "aa",
            "",
            "LC example 3 — not enough chars"
        ),
        (
            "AABBCC", "ABC",
            "ABBC",
            "duplicates in s"
        ),
        (
            "ABC", "ABC",
            "ABC",
            "exact match — full string"
        ),
        (
            "XYZABC", "CBA",
            "ABC",
            "t chars in reverse order in s"
        ),
        (
            "aa", "aa",
            "aa",
            "duplicate required chars"
        ),
        (
            "ZZZA", "B",
            "",
            "required char not in s"
        ),
    ]

    passed = 0
    for s, t, expected, label in tests:
        result = func(s, t)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"[{status}] {label}\n"
            f"         s={s!r}, t={t!r}\n"
            f"         expected={expected!r}, got={result!r}"
        )
    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def minWindow(s: str, t: str) -> str:
    """
    Find the shortest substring of s containing all chars of t.

    Approach:
    Build a frequency map of what t needs (need dict).
    Slide a window with left and right pointers across s.
    Expand right to add chars; track a 'formed' count for
    how many unique required chars are satisfied.
    When formed == required, shrink left to minimize window,
    saving the best result seen.

    Time:  O(|s| + |t|) — each char visited at most twice.
    Space: O(|s| + |t|) — two frequency maps.
    """
    pass


# Quick debug prints — run this cell while building
print(minWindow("ADOBECODEBANC", "ABC"))  # expected: "BANC"
print(minWindow("a", "a"))                # expected: "a"
print(minWindow("a", "aa"))               # expected: ""
print(minWindow("ABC", "ABC"))            # expected: "ABC"
print(minWindow("ZZZA", "B"))             # expected: ""

In [ ]:
# Uncomment and run when solution is ready
# test_harness(minWindow)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (all substrings) | O(n³) | O(1) |
| Sliding window (optimal) | O(\|s\| + \|t\|) | O(\|s\| + \|t\|) |

The sliding window avoids re-scanning substrings by keeping
live frequency counts — each character is added and removed
from the window at most once.

## Real World Connection

At Citi, a Kinesis stream carries events from 6,000+
endpoints — CPU, memory, disk, and network metrics.
Finding the minimum time window that contains at least
one reading from every critical service is the same
problem: a Lambda consumer uses a sliding window over
the event stream to detect the shortest observation
window where all required metric types appeared.
DynamoDB stores the window boundaries and CloudWatch
alarms trigger when that window exceeds an SLA threshold.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra